In [4]:
import pandas as pd
import os

In [9]:
#create a reuseable function

def process_vaers_year(year):

    # Load files 
    base_path = "E:/Project Challenge/ADEGuard/data/raw"
    
    data_path = os.path.join(base_path, f"{year}VAERSDATA.csv")
    symptoms_path = os.path.join(base_path, f"{year}VAERSSYMPTOMS.csv")
    vax_path = os.path.join(base_path, f"{year}VAERSVAX.csv")
    
    data = pd.read_csv(data_path, encoding="latin1", low_memory=False)
    symptoms = pd.read_csv(symptoms_path, encoding="latin1", low_memory=False)
    vax = pd.read_csv(vax_path, encoding="latin1", low_memory=False)

    # -------- Severity Normalize --------
    severity_cols = [
        "DIED", "L_THREAT", "ER_VISIT", "HOSPITAL",
        "DISABLE", "RECOVD", "OFC_VISIT",
        "ER_ED_VISIT", "BIRTH_DEFECT"
    ]

    for col in severity_cols:
        if col in data.columns:
            data[col] = data[col].fillna("N")
            data[col] = data[col].apply(
                lambda x: 1 if str(x).strip().upper() == "Y" else 0
            )

    # -------- DATA Aggregation --------
    data_clean = data.groupby("VAERS_ID").agg({
        "STATE": "first",
        "SEX": "first",
        "AGE_YRS": lambda x: x.dropna().iloc[0] if len(x.dropna()) > 0 else None,

        # Dates (preserved like manual version)
        "RECVDATE": "first",
        "RPT_DATE": "first",
        "VAX_DATE": "first",
        "ONSET_DATE": "first",

        # Severity flags
        "DIED": "max",
        "L_THREAT": "max",
        "ER_VISIT": "max",
        "HOSPITAL": "max",
        "DISABLE": "max",
        "RECOVD": "max",
        "OFC_VISIT": "max",
        "ER_ED_VISIT": "max",
        "BIRTH_DEFECT": "max",

        # Numeric
        "HOSPDAYS": "max",
        "NUMDAYS": "max",

        # Clinical Text
        "SYMPTOM_TEXT": lambda x: " ".join(x.dropna().unique()),
        "LAB_DATA": lambda x: " ".join(x.dropna().unique()),
        "OTHER_MEDS": lambda x: " ".join(x.dropna().unique()),
        "CUR_ILL": lambda x: " ".join(x.dropna().unique()),
        "HISTORY": lambda x: " ".join(x.dropna().unique()),
        "PRIOR_VAX": lambda x: " ".join(x.dropna().unique()),
        "ALLERGIES": lambda x: " ".join(x.dropna().unique())

    }).reset_index()

    # -------- Symptoms Aggregate --------
    symptom_cols = ["SYMPTOM1","SYMPTOM2","SYMPTOM3","SYMPTOM4","SYMPTOM5"]

    symptoms["ALL_SYMPTOMS"] = symptoms[symptom_cols] \
        .apply(lambda row: [x for x in row if pd.notnull(x)], axis=1)

    symptoms_agg = symptoms.groupby("VAERS_ID")["ALL_SYMPTOMS"] \
                            .sum() \
                            .reset_index()

    symptoms_agg["ALL_SYMPTOMS"] = symptoms_agg["ALL_SYMPTOMS"] \
        .apply(lambda x: list(set(x)))

    # -------- Vaccine Aggregate --------
    vax_subset = vax[["VAERS_ID","VAX_NAME","VAX_TYPE","VAX_MANU"]]

    vax_agg = vax_subset.groupby("VAERS_ID").agg({
        "VAX_NAME": lambda x: list(x.dropna().unique()),
        "VAX_TYPE": lambda x: list(x.dropna().unique()),
        "VAX_MANU": lambda x: list(x.dropna().unique())
    }).reset_index()

    # -------- Merge --------
    df = data_clean.merge(symptoms_agg, on="VAERS_ID", how="left")
    df = df.merge(vax_agg, on="VAERS_ID", how="left")

    df["YEAR"] = year

    return df

In [10]:
years = [2020, 2021, 2022, 2023, 2024, 2025]

all_data = []

for year in years:
    print(f"Processing {year}...")
    df_year = process_vaers_year(year)
    all_data.append(df_year)

df_master = pd.concat(all_data, ignore_index=True)

Processing 2020...
Processing 2021...
Processing 2022...
Processing 2023...
Processing 2024...
Processing 2025...


In [11]:
df_master.head()

,VAERS_ID,STATE,SEX,AGE_YRS,RECVDATE,RPT_DATE,VAX_DATE,ONSET_DATE,DIED,L_THREAT,...,OTHER_MEDS,CUR_ILL,HISTORY,PRIOR_VAX,ALLERGIES,ALL_SYMPTOMS,VAX_NAME,VAX_TYPE,VAX_MANU,YEAR
0,275438,CA,F,NaN,01/13/2020,None,03/12/2007,03/26/2007,1,0,...,YASMIN,,Medical History/Concurrent Conditions: Non-smo...,,,"[Pulmonary congestion, Death, Autopsy, Seizure...",[HPV (GARDASIL)],[HPV4],[MERCK & CO. INC.],2020
1,340381,PA,M,11.0,07/29/2020,None,02/13/2009,02/13/2009,0,0,...,INVEGA,Allergic rhinitis; Asthma (Mild persistent); A...,,,,"[Injection site erythema, Type III immune comp...","[MENINGOCOCCAL CONJUGATE (MENACTRA), TDAP (NO ...","[MNQ, TDAP]","[SANOFI PASTEUR, UNKNOWN MANUFACTURER]",2020
2,361824,NY,F,NaN,11/19/2020,None,None,None,0,0,...,,Drug allergy; Penicillin allergy,,,,"[Urticaria, Vaccination complication, Rash]",[INFLUENZA (SEASONAL) (FLUZONE HIGH-DOSE QUADR...,[FLU4],[SANOFI PASTEUR],2020
3,371224,CA,U,NaN,10/19/2020,None,None,None,0,0,...,,,,,,[Facial paralysis],[ZOSTER LIVE (ZOSTAVAX)],[VARZOS],[MERCK & CO. INC.],2020
4,386005,NM,F,NaN,01/13/2020,None,04/12/2010,None,0,0,...,,,,,,[Post herpetic neuralgia],[ZOSTER LIVE (ZOSTAVAX)],[VARZOS],[MERCK & CO. INC.],2020


In [12]:
df_master.shape

(1228214, 31)

In [15]:
df_master.to_csv("E:/Project Challenge/ADEGuard/data/processed/final_dataset_2020_2025.csv", index=False)

In [16]:
df_master.to_parquet("E:/Project Challenge/ADEGuard/data/processed/final_dataset_2020_2025.parquet", index=False)